In [ ]:
import torch
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

Using device: cpu


In [ ]:
import numpy as np
def generate_data(seq_length=5, num_samples=10000):
    x = np.linspace(0, 100, num_samples)
    y = np.sin(x)
    sequences = np.lib.stride_tricks.sliding_window_view(y, seq_length)
    labels = y[seq_length:]
    return sequences[:-1], labels
x,y = generate_data()
from sklearn.model_selection import train_test_split
x_tv,x_test,y_tv, y_test = train_test_split(x, y, test_size=0.2, random_state=42)
x_train,x_val,y_train, y_val = train_test_split(x_tv, y_tv, test_size=0.2, random_state=42)

In [ ]:
from torch.utils.data import Dataset,DataLoader
class RnnData(Dataset):
  def __init__(self,X,y):
    self.sequences = torch.from_numpy(X).float().unsqueeze(-1)
    self.labels = torch.from_numpy(y).float()
  def __len__(self):
    return len(self.labels)
  def __getitem__(self, idx):
    return {
        'input':self.sequences[idx],
        'label':self.labels[idx]
    }
train_dataset = RnnData(x_train,y_train)
val_dataset = RnnData(x_val,y_val)
test_dataset = RnnData(x_test,y_test)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)
sample = next(iter(train_loader))
print(f"Input shape: {sample['input'].shape}, Label shape: {sample['label'].shape}")

Input shape: torch.Size([32, 5, 1]), Label shape: torch.Size([32])


In [ ]:
from torch import nn
class RnnModel(nn.Module):
  def __init__(self,input,hidden,output):
    super(RnnModel,self).__init__()
    self.h = hidden
    self.rnn = nn.RNN(input_size=input,hidden_size=hidden,num_layers=1,batch_first=True)
    self.fc = nn.Linear(hidden,output)
  def forward(self,x):
    h0 = torch.zeros(1, x.size(0), self.h).to(x.device)
    x,_ = self.rnn(x,h0)
    x = self.fc(x[:,-1,:])
    return x.squeeze(-1)

In [ ]:
from torch.optim import Adam, SGD
i = 1
o =1
hidden = 32
model = RnnModel(i,hidden,o).to(device)
loss_fn = nn.MSELoss()
optimizer = Adam(model.parameters(),lr=0.001)

In [ ]:
epochs = 5
for i in range(epochs):
  model.train()
  total_loss,mse=0,0
  for batch in train_loader:
    inputs = batch['input'].to(device)
    labels = batch['label'].to(device)
    outputs = model(inputs)
    loss = loss_fn(outputs,labels)
    mse+=torch.square(outputs-labels).sum().item()
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    total_loss+=loss.item()
  rmse = (mse / n) ** 0.5
  model.eval()
  with torch.no_grad():
    total_vloss,n,mse,mae=0,0,0,0
    for batch in val_loader:
      inputs = batch['input'].to(device)
      labels = batch['label'].to(device)
      outputs = model(inputs)
      loss = loss_fn(outputs,labels)
      mae+=torch.abs(outputs-labels).sum().item()
      mse+=torch.square(outputs-labels).sum().item()
      n+=len(labels)
      total_vloss+=loss.item()
    mae /= n
    rmse_v = (mse / n) ** 0.5
    avg = total_vloss/len(test_loader)
    avg_loss= total_loss/len(train_loader)
  print(f"Epoch {i+1}/{epochs}, Train Loss: {avg_loss:.4f} Train rmse: {rmse:.3f} | Test Loss: {avg:.3f} , Test rmse: {rmse_v:.3f}")

Epoch 1/5, Train Loss: 0.0729 Train rmse: 0.483 | Test Loss: 0.001 , Test rmse: 0.035
Epoch 2/5, Train Loss: 0.0011 Train rmse: 0.066 | Test Loss: 0.001 , Test rmse: 0.031
Epoch 3/5, Train Loss: 0.0009 Train rmse: 0.060 | Test Loss: 0.001 , Test rmse: 0.028
Epoch 4/5, Train Loss: 0.0008 Train rmse: 0.055 | Test Loss: 0.001 , Test rmse: 0.025
Epoch 5/5, Train Loss: 0.0006 Train rmse: 0.050 | Test Loss: 0.000 , Test rmse: 0.024


In [ ]:
model.eval()
with torch.no_grad():
  total_loss,n,mse,mae=0,0,0,0
  for batch in test_loader:
    inputs = batch['input'].to(device)
    labels = batch['label'].to(device)
    outputs = model(inputs)
    loss = loss_fn(outputs,labels)
    mae+=torch.abs(outputs-labels).sum().item()
    mse+=torch.square(outputs-labels).sum().item()
    n+=len(labels)
    total_loss+=loss.item()
  mae /= n
  rmse = (mse / n) ** 0.5
  avg = total_loss/len(test_loader)
  print(f"Test mae: {mae:.2f} Test loss: {avg:.3f} Test rmse: {rmse:.2f}")
  print("Total_loss: ",total_loss)

Test mae: 0.02 Test loss: 0.001 Test rmse: 0.02
Total_loss:  0.037539931188803166


In [ ]:
from sklearn.metrics import r2_score
y_pred = model.forward(torch.tensor(x_test,device=device,dtype=torch.float32).unsqueeze(-1)).squeeze(-1).detach().cpu().numpy()
r2_score(y_pred,y_test)

#Manual RNN

Model Building:</br>
rnn formula is-</br>
h = tanh(X.Wxh + h_prev.Whh + bh) #It is calculated sequentially for each time step and weightare same for all time steps but different for each sample.</br>
output layer with linear activation formula :
y = H.Why + by

Now weights intilization, we use Xavier initialization specifically for tanh</br>
W - [ 0 , sqrt(1/n_input) ]</br>
H for 0th time step or 1st sequence is initialized 0 </br>
we then calculate the hidden states and use the last hidden state for output layer </br>



In [ ]:
class RnnScratch:
  def __init__(self,input,hidden,output):
    self.input = input
    self.hidden = hidden
    self.output = output
    self.wxh = torch.randn(self.input,self.hidden,device=device)*torch.sqrt(torch.tensor(1.0/self.input))
    self.whh = torch.randn(self.hidden,self.hidden,device=device)*torch.sqrt(torch.tensor(1.0/self.hidden))
    self.why = torch.randn(self.hidden,self.output,device=device)*torch.sqrt(torch.tensor(1.0/self.hidden))
    self.bh = torch.zeros(self.hidden,device=device)
    self.by = torch.zeros(self.output,device=device)
    #torch.nn.init.orthogonal_(self.whh)
    #torch.nn.init.xavier_uniform_(self.wxh)
    #torch.nn.init.xavier_uniform_(self.why)
    self.alpha = torch.nn.Parameter(torch.ones(1))
    self.layer_norm = torch.nn.LayerNorm(self.hidden)
    self.beta = torch.nn.Parameter(torch.tensor(0.5))
  def forward(self,x):
    h = torch.zeros(x.size(0),self.hidden,device=x.device)
    self.h_states = []
    dropout = torch.nn.Dropout(0.2)  # Use dropout in hidden state
    for i in range(x.size(1)):
      h = torch.tanh_(x[:, i, :] @ self.wxh + h @ self.whh + self.bh.unsqueeze(0))
      #h = torch.nn.functional.leaky_relu_(x[:, i, :] @ self.wxh + h @ self.whh + self.bh.unsqueeze(0))+x[:,i,:]
      #h = torch.nn.functional.leaky_relu_(x[:, i, :] @ self.wxh + h @ self.whh + self.bh.unsqueeze(0)) + self.alpha * x[:, i, :]
      #h = dropout(torch.tanh_(x[:, i, :] @ self.wxh + h @ self.whh + self.bh.unsqueeze(0))) + self.alpha * x[:, i, :]
      #h = self.layer_norm(torch.tanh_(x[:, i, :] @ self.wxh + h @ self.whh + self.bh.unsqueeze(0))) + self.alpha * x[:, i, :]
      #h = (1 - self.beta) * h + self.beta * self.layer_norm(torch.relu(x[:, i, :] @ self.wxh + h @ self.whh + self.bh.unsqueeze(0)))
      #h = dropout(h) + + self.alpha * x[:, i, :]
      self.h_states.append(h)
    self.h_states = torch.stack(self.h_states, dim=1)
    y = h@self.why+self.by.unsqueeze(0)
    return y

In [ ]:
i = 1
o =1
hidden = 32
model = RnnScratch(i,hidden,o)

We implement RNN training using backpropogation through time</br>
for a single layer rnn for dynamic hidden size, we used the chain rule to derive formulaes and generalize them to be used for propogating errors backwards in time.</br>
for by and wyh, formulaes are straightforward since they don't involve time.</br>
for bh, wxh, whh, we need to propogate error backwards, simple saying, our loss function L depends on not only one hidden state, but all hidden states, because future hidden states depend on prev hidden states in formula</br>
so we calculate dL/dH for each time step in backward loop</br>
for bh, we find dL/dbh which is just sum of errors of dL/dH</br>
for whh, its the errors dL/dH multiplied with hidden state</br>
for wxh, its the errors dL/dH multiplied with input sequences</br>
then we used SGD to update the weights, and calculated loss and checked with validation test, and finally with test set and calculated r2_score

In [ ]:
lr_rate = 0.001
epochs=5
for epoch in range(epochs):
  total_loss=0

  for batch in train_loader:
      inputs = batch['input'].to(device)
      labels = batch['label'].unsqueeze(-1).to(device)
      outputs = model.forward(inputs)
      loss = torch.mean((outputs - labels) ** 2)
      grad_ly = 2 * (outputs - labels)
      grad_by = torch.sum(grad_ly, dim=0)
      grad_lh = grad_ly @ model.why.T
      grad_why = model.h_states[:, -1, :].T @ grad_ly
      grad_wxh = 0
      grad_wbh = 0
      grad_whh = 0
      for i in range(model.h_states.size(1) - 1, -1, -1):
          grad_lh = (1 - model.h_states[:, i, :] ** 2) * (grad_lh @ model.whh.T)
          #grad_lh = torch.clamp(grad_lh, min=-5, max=5)
          grad_wxh += inputs[:, i, :].T @ grad_lh
          grad_wbh += grad_lh.sum(dim=0)
          if i > 0:
              grad_whh += model.h_states[:, i - 1, :].T @ grad_lh
      #torch.nn.utils.clip_grad_norm_([grad_wxh, grad_whh, grad_why,grad_lh], max_norm=1.0)
      model.by -= lr_rate * grad_by
      model.whh -= lr_rate * grad_whh
      model.why -= lr_rate * grad_why
      model.wxh -= lr_rate * grad_wxh
      model.bh -= lr_rate * grad_wbh
      total_loss+=loss.item()
  avg_loss= total_loss/len(train_loader)
  total_vloss,n,mse,mae=0,0,0,0
  for batch in val_loader:
    inputs = batch['input'].to(device)
    labels = batch['label'].to(device)
    outputs = model.forward(inputs)
    preds = outputs.squeeze(-1)
    mae+=torch.abs(preds-labels).sum().item()
    mse+=torch.square(preds-labels).sum().item()
    n+=len(labels)
    loss = torch.mean((outputs - labels) ** 2)
    total_vloss+=loss.item()
  mae /= n
  rmse = (mse / n) ** 0.5
  avg = total_vloss/len(test_loader)
  print(f"Epoch {epoch+1}/{epochs}, Train Loss: {avg_loss:.4f}")
  print(f"Val mae: {mae:.2f} Val loss: {avg:.3f} Val rmse: {rmse:.2f}")

Epoch 1/5, Train Loss: 0.0014
Val mae: 0.01 Val loss: 0.794 Val rmse: 0.02
Epoch 2/5, Train Loss: 0.0002
Val mae: 0.01 Val loss: 0.791 Val rmse: 0.01
Epoch 3/5, Train Loss: 0.0001
Val mae: 0.01 Val loss: 0.795 Val rmse: 0.01
Epoch 4/5, Train Loss: 0.0001
Val mae: 0.01 Val loss: 0.793 Val rmse: 0.01
Epoch 5/5, Train Loss: 0.0001
Val mae: 0.01 Val loss: 0.793 Val rmse: 0.01


In [ ]:
total_loss,n,mse,mae=0,0,0,0
for batch in test_loader:
  inputs = batch['input'].to(device)
  labels = batch['label'].to(device)
  outputs = model.forward(inputs)
  preds = outputs.squeeze(-1)
  mae+=torch.abs(preds-labels).sum().item()
  mse+=torch.square(preds-labels).sum().item()
  n+=len(labels)
  loss = torch.mean((outputs - labels) ** 2)
  total_loss+=loss.item()
mae /= n
rmse = (mse / n) ** 0.5
avg = total_loss/len(test_loader)
print(f"Test mae: {mae:.2f} Test loss: {avg:.3f} Test rmse: {rmse:.2f}")

Test mae: 0.01 Test loss: 0.961 Test rmse: 0.01


In [ ]:
from sklearn.metrics import r2_score
y_pred = model.forward(torch.tensor(x_test,device=device,dtype=torch.float32).unsqueeze(-1)).squeeze(-1).detach().cpu().numpy()
r2_score(y_pred,y_test)

0.9998879249826252